# News IR: complete E5 encoding on a Kaggle GPU

Attach the private Dataset made from `artifacts/bundles/news_ir_kaggle.zip`. Enable **Internet** and an available **GPU** under Settings. No API key is required. This runs the existing project implementation over the same 594,589 eligible articles as BM25.

Set the budget below, then use **Save Version → Save & Run All** to run once in the background and save outputs when it finishes. Allow time beyond the encoding budget for setup, indexing and output saving. A six-hour encoding budget is a starting value, not a guarantee about quota or runtime. Check your account's remaining allowance first.


In [ ]:
# Change these values before starting the saved run.
BATCH_SIZE = 32  # Reduce to 16 or 8 if CUDA reports out of memory.
TIME_BUDGET_HOURS = 6  # Encoding only; keep below the remaining session/quota allowance.
# For a new session, attach the previous saved notebook output (or upload its checkpoint
# ZIP as another private Dataset) and set its path here. Leave None on your first run.
RESUME_ARCHIVE = None
# Example: "/kaggle/input/your-previous-output/dense-checkpoints.zip"


## Install the encoding dependencies and check the allocated GPU

This can take several minutes. A separate environment avoids conflicts with Kaggle's host packages. It uses the tested model-library versions and the official PyTorch 2.9.1 CUDA 12.6 wheel. The actual runtime is recorded with every shard. Model and corpus downloads are public, with token use disabled. This job uses one GPU (`cuda:0`), including when Kaggle offers T4 x2. The setup runs a real CUDA matrix operation before the full job starts.

Environment creation skips `ensurepip` and uses the official PyPA pip installer, so it also works when the hosted Python image cannot bootstrap pip through `venv`.


In [ ]:
import os, subprocess, sys, urllib.request
from pathlib import Path
os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
# A clean environment avoids conflicts with Kaggle's unrelated preinstalled packages.
venv = Path("/kaggle/temp/news_ir_venv")
venv.parent.mkdir(parents=True, exist_ok=True)
# Skip ensurepip: it can be unavailable in hosted Python images.
subprocess.run([sys.executable, "-m", "venv", "--without-pip", str(venv)], check=True)
GPU_PYTHON = str(venv / "bin/python")
# Bootstrap pip inside this environment using the official installer.
bootstrap_pip = venv.parent / "news_ir_get_pip.py"
urllib.request.urlretrieve("https://bootstrap.pypa.io/get-pip.py", bootstrap_pip)
subprocess.run([GPU_PYTHON, str(bootstrap_pip), "--quiet"], check=True)
# Select the official CUDA 12.6 wheel explicitly; do not replace Kaggle's host packages.
subprocess.run([GPU_PYTHON, "-m", "pip", "install", "--quiet",
                "torch==2.9.1", "--index-url", "https://download.pytorch.org/whl/cu126"], check=True)
subprocess.run([GPU_PYTHON, "-m", "pip", "install", "--quiet",
                "transformers==5.20.1", "huggingface-hub==2.2.0",
                "faiss-cpu==1.13.2", "numpy>=2,<3"], check=True)
subprocess.run([GPU_PYTHON, "-c", "import torch; "
                "assert torch.cuda.is_available(), 'Enable GPU under Kaggle Settings'; "
                "print('GPU:', torch.cuda.get_device_name(0)); "
                "print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory/1024**3, 1)); "
                "x=torch.ones((16,16), device='cuda'); print('CUDA kernel check:', (x@x)[0,0].item())"],
               check=True)


## Find the attached bundle and run the complete job

Kaggle may leave your ZIP intact or automatically extract it; both are supported. Logs show verified downloads, a 128-article throughput estimate, and progress after every 2,048-article shard. The estimate excludes setup, checkpoint export and indexing. A timed stop preserves the full-corpus target and exports a checkpoint; it does not create a pilot experiment.


In [ ]:
import json, tempfile, zipfile
from pathlib import Path

input_root = Path("/kaggle/input")
candidates = sorted(input_root.rglob("news_ir_kaggle.zip"))
candidates += sorted(p.parent for p in input_root.rglob("bundle.json")
                     if (p.parent / "scripts/run_kaggle_dense.py").is_file())
if len(candidates) != 1:
    raise RuntimeError(f"Expected one attached project bundle; found {len(candidates)}. "
                       "Use Add Input to attach your private project Dataset.")
bundle = candidates[0]
bootstrap = Path(tempfile.mkdtemp(prefix="news-ir-launch-")) / "run_kaggle_dense.py"
if bundle.is_dir():
    bootstrap.write_bytes((bundle / "scripts/run_kaggle_dense.py").read_bytes())
else:
    with zipfile.ZipFile(bundle) as archive:
        bootstrap.write_bytes(archive.read("scripts/run_kaggle_dense.py"))
command = [GPU_PYTHON, "-u", str(bootstrap), "--bundle", str(bundle),
           "--batch-size", str(BATCH_SIZE), "--time-budget-hours", str(TIME_BUDGET_HOURS)]
if RESUME_ARCHIVE:
    command += ["--resume-archive", str(RESUME_ARCHIVE)]
subprocess.run(command, check=True)


## Read the saved outcome

When **completed** is true, download `dense-results.zip` and `job_summary.json` from the saved version's **Output** tab. Put the results ZIP in the local IR project folder. When false, save/download `dense-checkpoints.zip`, attach it to the next GPU notebook as input, set `RESUME_ARCHIVE` above and run again. A checkpoint contains finished shards; a hard session termination can lose outputs that have not been saved. Keep a saved checkpoint before spending another session.

The final result is about 1.7 GiB, plus a similarly sized checkpoint. Full encoding, index and archive copies need roughly 7 GiB of working-output space; temporary downloads need additional space outside that output directory. Do not upload model weights or the full corpus manually.


In [ ]:
summary = json.loads(Path("/kaggle/working/job_summary.json").read_text())
print(json.dumps(summary, indent=2))
if summary["completed"]:
    assert summary["encoded_documents"] == summary["target_documents"] == 594589
    print("COMPLETE: download dense-results.zip and job_summary.json from the saved Output tab.")
else:
    print("CHECKPOINT ONLY: save dense-checkpoints.zip and resume in another GPU session.")


## Back on the Mac

From the project directory:

```sh
unzip -n dense-results.zip -d artifacts
.venv/bin/news-ir dense-search "cricket insect lifecycle" --top-k 5
```

Retain the saved Kaggle checkpoints until the local index is verified. This notebook exports rankings for the 20 **draft** queries. Query review, pooled human relevance judgments, nDCG@5, MAP, failure analysis and the research report remain separate project work. No model training or LLM API charges are involved.
